# 4. Equilibration with GROMACS

We put an SDS micelle (a ball) in water, inside a project, and equilibrate it with GROMACS. The equilibration runs in stages:

- **stage 0**: energy minimisation, about 1–2 minutes on Colab
- **stage 1**: short MD with position restraints, about 20 minutes on Colab, if there is time.

Then we look at the outputs, save the project, and export the stages to open them in VMD.

Run the setup cell first. On Colab it installs GROMACS and Shapespyer, which takes about 3 minutes. To keep the project after the Colab runtime ends, tick `SAVE_PROJECTS_TO_DRIVE` before you run it: Colab then asks you once to sign in to Google Drive. Then run the helpers cell under it: it only shows and downloads results, and isn't part of the Shapespyer workflow.

In [ ]:
# @title Setup: on Colab, install GROMACS and Shapespyer (about 3 min) { display-mode: "form" }
SAVE_PROJECTS_TO_DRIVE = False  # @param {type:"boolean"}

import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
CONFIG_FILE = Path("shapespyer.yaml")  # where Shapespyer keeps its files

if IN_COLAB:
    !apt-get -qq update > /dev/null && apt-get -qq install -y --no-install-recommends gromacs > /dev/null
    %pip install -q "shapespyer==0.3.0"

    # nglview draws molecules through Colab's widget manager
    from google.colab import output
    output.enable_custom_widget_manager()

    from shapes.project import Config
    config = Config(CONFIG_FILE)
    # the library and Temp stay on the fast local disk; /content shows in the Files panel
    config.app_path = "/content/shapespyer"
    # projects go there too, or to Drive, where they stay after the runtime ends
    config.projects_path = None
    if SAVE_PROJECTS_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")  # asks once for Google sign-in
        config.projects_path = "/content/drive/MyDrive/shapespyer/Projects"
    config.save()
elif not CONFIG_FILE.is_file():
    err_msg = f"No {CONFIG_FILE} in {Path.cwd()}; run the notebook in its own folder, next to shapespyer.yaml"
    raise FileNotFoundError(err_msg)

from shapes.project import Config
from shapes.project.files import AppFileManager

# the first call sets where the files go, for the whole session
config = AppFileManager(CONFIG_FILE).config
if config.projects_path != Config(CONFIG_FILE).projects_path:
    err_msg = "Shapespyer already runs with other folders: Runtime > Restart session, then run this cell again"
    raise RuntimeError(err_msg)
print("Shapespyer files:", config.app_path, "| projects:", config.projects_path)
if config.gmx_problem is not None:
    err_msg = (
        f"This notebook needs GROMACS: {config.gmx_problem}. Install it, or set gmx_executable"
        f" in {CONFIG_FILE}. On Windows, the easy way is to run this notebook in WSL or on Colab."
    )
    raise RuntimeError(err_msg)
print("GROMACS:", config.gmx_version)

In [ ]:
# @title Helpers (not part of the Shapespyer workflow) { display-mode: "form" }
import re
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import mdtraj
import nglview
import numpy as np


def show_mdp_settings(configuration):
    """Print the main settings of each equil<k>.mdp, to compare the stages."""
    print(f"{'file':<12}{'integrator':<12}{'nsteps':>9}{'dt (ps)':>9}{'time (ps)':>11}  define")
    for mdp in configuration.paths:
        found = re.findall(r"^\s*([\w-]+)\s*=\s*([^;\n]*?)\s*(?:;.*)?$", mdp.read_text(), re.MULTILINE)
        settings = dict(found)
        nsteps, dt = settings.get("nsteps", ""), settings.get("dt", "")
        time_ps = f"{int(nsteps) * float(dt):g}" if nsteps and dt else ""
        integrator = settings.get("integrator", "")
        print(f"{mdp.name:<12}{integrator:<12}{nsteps:>9}{dt:>9}{time_ps:>11}  {settings.get('define', '')}")


def show_minimisation_result(stage):
    """Print the lines of a minimisation stage's .log that tell whether it converged."""
    for line in _file_with_suffix(stage, ".log").read_text().splitlines():
        if "converged" in line or "Potential Energy" in line or "Maximum force" in line:
            print(line)


def _file_with_suffix(asset, suffix):
    found = [path for path in asset.paths if path.suffix == suffix]
    if not found:
        raise FileNotFoundError(f"{asset.name} holds no {suffix} file yet")
    return found[0]


def _molecules(topology, resname):
    """Atom indices of the residues named resname (a name or a list of names), or of all but the water."""
    # mdtraj renames TIP3 to HOH on loading, so select it with the "water" keyword
    if resname is None:
        return topology.select("not water")
    names = [resname] if isinstance(resname, str) else list(resname)
    atoms = topology.select(" or ".join(f"resname {name}" for name in names))
    if len(atoms) == 0:
        found = sorted({residue.name for residue in topology.residues})
        raise ValueError(f"No residues named {', '.join(names)}; the file has {', '.join(found)}")
    return atoms


def show_without_water(asset, resname=None):
    """Draw the .gro an asset holds, such as a sample model or a stage, without the water.

    Give resname, a residue name or a list of them, to draw only those molecules.
    """
    frame = mdtraj.load(_file_with_suffix(asset, ".gro"))
    shown = frame.atom_slice(_molecules(frame.top, resname))
    print(f"{frame.n_atoms} atoms, box {frame.unitcell_lengths[0]} nm; {shown.n_atoms} shown")
    return nglview.show_mdtraj(shown)


def _load_trajectory(stage):
    return mdtraj.load(_file_with_suffix(stage, ".xtc"), top=_file_with_suffix(stage, ".gro"))


def show_trajectory_without_water(stage, resname=None):
    """Draw the trajectory of an MD stage, without the water; resname draws only those molecules."""
    trajectory = _load_trajectory(stage)
    return nglview.show_mdtraj(trajectory.atom_slice(_molecules(trajectory.top, resname)))


def plot_micelle_rg(stage, resname):
    """Plot the radius of gyration of the residues named resname, per frame of an MD stage."""
    trajectory = _load_trajectory(stage)
    micelle = trajectory.atom_slice(trajectory.top.select(f"resname {resname}"))
    masses = np.array([atom.element.mass for atom in micelle.top.atoms])
    rg = mdtraj.compute_rg(micelle, masses=masses)  # nm, one value per frame
    print(f"{micelle.n_frames} frames, Rg from {rg.min():.2f} to {rg.max():.2f} nm")
    plt.plot(micelle.time, rg)
    plt.xlabel("time (ps)")
    plt.ylabel("Rg of the micelle (nm)")
    plt.show()


def show_stages(equilibration):
    """Print each stage of an equilibration with the names of its files."""
    for stage in equilibration.stages:
        print(stage.name, [path.name for path in stage.paths])


def list_files(folder):
    """Print the files under a folder, relative to it."""
    for path in sorted(folder.rglob("*")):
        if path.is_file():
            print(path.relative_to(folder))


def download_as_zip(folder):
    """Zip a folder into <folder>.zip next to it; on Colab, also download the zip."""
    archive = Path(shutil.make_archive(str(folder), "zip", folder))
    if IN_COLAB:
        from google.colab import files
        files.download(str(archive))
    return archive

In [ ]:
import shutil
import time

from shapes.basics.utils import LogConfiguration
from shapes.project import Ball, Ion, Library, Project, Surfactant
from shapes.project.analysis import AnalysisConfiguration
from shapes.project.species import Solvent

LogConfiguration("WARNING")

library = Library()
library.clean_temp()

# a new folder each run, so a project saved before, such as one in Drive, is kept
DEMO_PATH = config.projects_path / f"DemoProj-{time.strftime('%Y%m%d-%H%M%S')}"
# the stage exports and their zip go to a work folder, which the clean up deletes
WORK = config.app_path / "work"

### Create the project and its sample model

The project goes into the projects folder that the setup cell printed: `.data/Projects` locally, `/content/shapespyer/Projects` on Colab, or your Drive. The path given to `Project` is only noted: the folder is made on the first save.

SDS is charged, so it gets sodium (NAT) as its counter ion.

In [ ]:
proj = Project(path=DEMO_PATH)
model = proj.sample_model
model.name = "SDS_ball"

ball = Ball(name="SDS_micelle")
ball.layers[0].dmin = 0.5
ball.layers[0].rmin = 1.2

sds = Surfactant.load("SDS")
sds.mint = 41
sds.counter_ion = Ion.load("NAT")
ball.components.append(sds)

model.structure = ball
model.buffer.solvent = Solvent.load("TIP3")

### Assemble the sample model

The assembly builds the ball, puts in one Na⁺ per SDS with `gmx insert-molecules`, and fills the box with water with `gmx solvate`. The files are made in Temp, in the application directory.

In [ ]:
model.assemble()
for path in model.paths:
    print(path)

### Set up the equilibration

Stage *k* runs `equil<k>.mdp`, starting from the final `.gro` of stage *k − 1*. The library's CHARMM36 set has 7 stages:

In [ ]:
equilibration = proj.analysis.equilibration
equilibration.configuration = AnalysisConfiguration.load("CHARMM36")
show_mdp_settings(equilibration.configuration)

### Stage 0: energy minimisation

`first_stage` and `last_stage` choose the stages to run. `.run()` waits until GROMACS ends.

In [ ]:
equilibration.first_stage = 0
equilibration.last_stage = 0
equilibration.run(model)

for path in equilibration.stages[0].paths:
    print(path)

The files of a stage:

- `.tpr`: the run input, made by `gmx grompp`
- `.log`: the log of `gmx mdrun`
- `.edr`: energies
- `.trr`: trajectory (`.xtc` for MD stages)
- `.gro`: final coordinates; the next stage starts from them

Did the minimisation converge?

In [ ]:
show_minimisation_result(equilibration.stages[0])

The minimised micelle. `resname="SDS"` draws only the SDS, without the water and the Na⁺ ions:

In [ ]:
show_without_water(equilibration.stages[0], resname="SDS")

### Stage 1, if there is time: short MD

Stage 1 runs MD with position restraints. It takes about 20 minutes on Colab, and the cell shows nothing until GROMACS ends. A run that ends is published to `equilibration.stages[1]`. Without the time, skip this cell and the trajectory and Rg cells after it. With *Run all*, the notebook waits here until stage 1 ends.

To stop it early, interrupt the cell: the stop button on Colab, *Interrupt* in Jupyter or VS Code. That stops GROMACS too. Stage 0 keeps its files; skip the trajectory and Rg cells.

In [ ]:
equilibration.first_stage = 1
equilibration.last_stage = 1
equilibration.run(model)

for path in equilibration.stages[1].paths:
    print(path)

The stage 1 trajectory of the micelle, SDS only:

In [ ]:
show_trajectory_without_water(equilibration.stages[1], resname="SDS")

How big is the micelle during stage 1? The radius of gyration (Rg), with the real masses of the atoms, gives one value per frame. If Rg stays about the same, the micelle keeps its size.

In [ ]:
plot_micelle_rg(equilibration.stages[1], resname="SDS")

### Save the project

`save()` makes the project folder and copies into it the sample model, the analysis and every asset in them, with the stages. `ser_data.yaml` records the whole project, and `manifest.yaml` lists the files.

In [ ]:
proj.save()
list_files(proj.path)

`Project.load` restores it from the folder, with the stages:

In [ ]:
loaded = Project.load(DEMO_PATH)
show_stages(loaded.analysis.equilibration)

### Export the stages and open them in VMD

The saved project is for Shapespyer: load it to go on with it. To look at the stages in VMD, export them: `export_to()` copies the files of a stage into a folder you choose, here `stages/eq<k>/` in the work folder. Then the cell zips them; on Colab it also downloads the zip. Unzip it on your computer and open a stage in VMD from its folder, for example:

```
vmd SDS_ball-eq0.gro
vmd SDS_ball-eq1.gro SDS_ball-eq1.xtc
```

In [ ]:
STAGE_EXPORTS = WORK / "stages"
for stage in equilibration.stages:
    stage.export_to(STAGE_EXPORTS / stage.name, overwrite=True)
download_as_zip(STAGE_EXPORTS)

### Clean up (optional)

This deletes the saved project, the work folder with the stage exports and their zip, and Temp, where the model and the stages were made. `force=True` is needed, because the assets of this session still hold their files there. Skip it to keep the project, for example in Drive.

In [ ]:
project_dir = proj.path
library.delete_project(project_dir, force=True)
shutil.rmtree(WORK, ignore_errors=True)
library.clean_temp(force=True)
print(project_dir.exists(), WORK.exists())